# TotalSegmentator Tutorial for PyTheranostics

This tutorial demonstrates how to use TotalSegmentator within PyTheranostics for automated CT segmentation and RT-STRUCT creation.

## About TotalSegmentator

TotalSegmentator is a powerful deep learning tool for automatic segmentation of 104 anatomical structures in CT images. PyTheranostics provides a streamlined interface to use TotalSegmentator for theranostics workflows.

**Citation**: 

> Wasserthal J, Breit HC, Meyer MT, et al. TotalSegmentator: Robust Segmentation of 104 Anatomic Structures in CT Images. *Radiology: Artificial Intelligence*. 2023;5(5):e230024. [https://doi.org/10.1148/ryai.230024](https://pubs.rsna.org/doi/10.1148/ryai.230024)

## Overview

This tutorial covers:
1. **Downloading example data** from the SNMMI Dosimetry Challenge
2. **Running TotalSegmentator** to create segmentation masks (`.nii.gz` files)
3. **Converting masks to RT-STRUCT** (DICOM format) with optional filtering/grouping
4. **Using configuration files** to customize VOIs within the RT-STRUCT.

---

In [1]:
from pytheranostics.data_fetchers import fetch_snmmi_dosimetry_challenge,get_data_dir
from pytheranostics.segmentation import totalseg_segment, convert_masks_to_rtstruct
from pathlib import Path

%reload_ext autoreload
%autoreload 2

## Quick Start: Initialize a New Project

Before starting, you can use PyTheranostics' project initialization tool to set up a standardized project structure with configuration templates:

```python
from pytheranostics import init_project

# Create a new project with all templates and standard directories
init_project("./my_dosimetry_study")
```

This creates:
- `total_seg_config.json` - TotalSegmentator configuration template
- `voi_mappings_config.json` - VOI name mapping template  
- Standard directories: `data/`, `results/`, `segmentations/`, `rtstructs/`, `notebooks/`
- `README.md` with project documentation

You can then customize the config files for your specific needs!

---

In [2]:
from pytheranostics import init_project

# Create a new project with all templates and standard directories
init_project("./my_dosimetry_project")

Initializing PyTheranostics project: /Users/curibe/Documents/pytheranostics/my_dosimetry_project
✓ Created README.md

✓ Project initialized: /Users/curibe/Documents/pytheranostics/my_dosimetry_project

Configuration files:
  ✓ total_seg_config.json
    └─ TotalSegmentator ROI filtering/renaming/combining
  ✓ voi_mappings_config.json
    └─ VOI name mappings for CT/SPECT analysis

Directories created:
  ✓ data/
  ✓ results/
  ✓ segmentations/
  ✓ rtstructs/
  ✓ notebooks/

Next steps:
  1. Edit configuration files to match your project needs
  2. Place DICOM data in data/ directory
  3. Run segmentation and analysis workflows



PosixPath('/Users/curibe/Documents/pytheranostics/my_dosimetry_project')

## Step 1: Download Example Data

We'll use the SNMMI Dosimetry Challenge dataset from University of Michigan Deep Blue. This contains multi-timepoint SPECT/CT data and we will focus on Patient_004.

In [3]:
fetch_snmmi_dosimetry_challenge()

Extracting...
Extraction complete ✓

Data ready at: /Users/curibe/.pytheranostics_example_data/snmmi_dose_challenge

Dataset citation:
  SNMMI Lu-177 Dosimetry Challenge Dataset
  Creators: Dewaraja, Yuni K and Van, Benjamin J
  DOI: https://doi.org/10.7302/864r-tb45
  Repository: University of Michigan Deep Blue


## Step 2: Define Output Directories

Set up folders for where TotalSegmentator will write segmentation masks and where RT-STRUCT files will be saved.

In [4]:
example_data_dir = get_data_dir()

# Provide a ROOT folder where all CT series under it will be discovered automatically.
# In this case, as an example, we provide the path to a single CT series at the first scan of Patient_004 from the downloaded dataset.
project_data_dir = example_data_dir / 'snmmi_dose_challenge/Patient_004/SPECT_Cts/scan1/ct'

# Specify paths of Where to write segmentations and RT-STRUCT outputs (will be grouped by PatientID)
totalsegmentator_output_dir = Path('./my_dosimetry_project/Segmentations')
rtstruct_output_dir = Path('./my_dosimetry_project/rtstructs')

## Step 3: Run TotalSegmentator

The `totalseg_segment()` function:
- **Discovers all CT series** under the `root_dir` automatically
- **Extracts patient ID** from DICOM metadata
- **Identifies timepoints** from folder structure (e.g., `scan1`, `scan2`)
- **Runs TotalSegmentator** to generate 104 segmentation masks per CT series
- **Returns both** segmentation paths and CT paths for later use

**Note**: This step can take several minutes per CT scan depending on your hardware.

In [5]:
# Run segmentation 
seg_result = totalseg_segment(
    root_dir=str(project_data_dir),
    base_output_dir=str(totalsegmentator_output_dir),
    device="mps", # Change to "cuda" if using an NVIDIA GPU, or "cpu" to run on CPU
    parallel=True,
    max_workers=4, # Number of parallel workers to use if parallel=True
)

Using device: MPS
Segmentation: patient=ANON54121 timepoint=scan1
  CT=/Users/curibe/.pytheranostics_example_data/snmmi_dose_challenge/Patient_004/SPECT_Cts/scan1/ct
  OUT=my_dosimetry_project/Segmentations/ANON54121/scan1

If you use this tool please cite: https://pubs.rsna.org/doi/10.1148/ryai.230024

Converting dicom to nifti...
  found image with shape (512, 512, 130)
Resampling...
  Resampled in 4.65s
Predicting part 1 of 5 ...


100%|██████████████████████████████████████████| 48/48 [00:18<00:00,  2.55it/s]


Predicting part 2 of 5 ...


100%|██████████████████████████████████████████| 48/48 [00:18<00:00,  2.55it/s]


Predicting part 3 of 5 ...


...

Predicting part 4 of 5 ...


100%|██████████████████████████████████████████| 48/48 [00:18<00:00,  2.58it/s]


Predicting part 5 of 5 ...


100%|██████████████████████████████████████████| 48/48 [00:18<00:00,  2.56it/s]


  Predicted in 149.24s
Resampling...
Saving segmentations...
  Saved in 6.63s


### Segmentation Output Structure

TotalSegmentator creates one `.nii.gz` file per anatomical structure (104 total). These are organized by patient ID and timepoint:

```
my_dosimetry_project/Segmentations/
└── ANON54121/
    └── scan1/
        ├── adrenal_gland_left.nii.gz
        ├── adrenal_gland_right.nii.gz
        ├── aorta.nii.gz
        ├── brain.nii.gz
        ├── heart.nii.gz
        ├── kidney_left.nii.gz
        ├── kidney_right.nii.gz
        ├── liver.nii.gz
        ├── lung_lower_lobe_left.nii.gz
        ├── lung_upper_lobe_left.nii.gz
        ├── rib_left_1.nii.gz
        ├── rib_left_2.nii.gz
        ... (104 structures total)
        └── vertebrae_T9.nii.gz
```

Each `.nii.gz` file is a 3D binary mask aligned with the original CT scan.

## Step 4: Convert to RT-STRUCT Format

### What is RT-STRUCT?

RT-STRUCT (Radiotherapy Structure Set) is a DICOM format for storing organ contours and regions of interest. It's the standard format used by:
- Treatment planning systems (Eclipse, RayStation, Monaco, etc.)
- DICOM viewers (MIM, 3D Slicer, etc.)
- Dosimetry calculation tools

Converting TotalSegmentator masks to RT-STRUCT allows you to:
- **Visualize** segmentations in clinical DICOM viewers
- **Import** into treatment planning systems
- **Use** for absorbed dose calculations and analysis
- **Share** with collaborators using standard DICOM format

### Using Configuration Files

By default, converting all 104 structures would create a very large RT-STRUCT file. Configuration files let you:
- **Filter**: Include only the organs you need (e.g., kidneys, liver, spleen)
- **Rename**: Change organ names to match your workflow conventions
- **Combine**: Merge multiple structures into one (e.g., all ribs → "ribs")


### Configuration File Format Explained

The JSON configuration has two main sections:

**1. `vois` (Volumes of Interest)**
- Lists individual TotalSegmentator structures
- `voi_name`: Must match the `.nii.gz` filename (without extension)
- `include`: Set to `true` to include this structure
- `new_name`: (Optional) Rename the structure in the RT-STRUCT

**2. `combine` (Optional)**
- Merges multiple structures into a single ROI
- `combined_voi_name`: Name for the merged ROI
- `sources`: List of `voi_name` values to combine (uses logical OR)

**Important**: When using `combine`, the individual source structures will NOT appear as separate ROIs - only the combined version will be included.


### Modifying the Configuration File

The `init_project()` method creates a default `total_seg_config.json` template in your project directory. You can open this file in VS Code (or any text editor) and customize which structures to include and how to process them.

**Open the file:** `ribs_kidneys_liver_project/total_seg_config.json`

**Common modifications:**
- Change `"include": true` to `"include": false` to exclude structures
- Add a `"new_name"` field to rename structures in the RT-STRUCT
- Define `"combine"` rules to merge related structures (e.g., all ribs into one ROI)

### Example 1: Filter to Kidneys and Liver Only

Simply edit your `total_seg_config.json` in VS Code to keep only these organs:

```json
{
  "vois": [
    {"voi_name": "kidney_left", "include": true, "new_name": "Left Kidney"},
    {"voi_name": "kidney_right", "include": true, "new_name": "Right Kidney"},
    {"voi_name": "liver", "include": true, "new_name": "Liver"}
  ],
  "combine": []
}
```

This config will create an RT-STRUCT with only three ROIs: Left Kidney, Right Kidney, and Liver.

### Example 2: Combine Ribs and Select Other Organs

For a bone marrow dosimetry study, you might want to combine all ribs into a single ROI. Open `total_seg_config.json` in VS Code and modify it like this:

```json
{
  "vois": [
    {"voi_name": "rib_left_1", "include": true},
    {"voi_name": "rib_left_2", "include": true},
    {"voi_name": "rib_left_3", "include": true},
    {"voi_name": "rib_left_4", "include": true},
    {"voi_name": "rib_left_5", "include": true},
    {"voi_name": "rib_left_6", "include": true},
    {"voi_name": "rib_left_7", "include": true},
    {"voi_name": "rib_left_8", "include": true},
    {"voi_name": "rib_left_9", "include": true},
    {"voi_name": "rib_left_10", "include": true},
    {"voi_name": "rib_left_11", "include": true},
    {"voi_name": "rib_left_12", "include": true},
    {"voi_name": "rib_right_1", "include": true},
    {"voi_name": "rib_right_2", "include": true},
    {"voi_name": "rib_right_3", "include": true},
    {"voi_name": "rib_right_4", "include": true},
    {"voi_name": "rib_right_5", "include": true},
    {"voi_name": "rib_right_6", "include": true},
    {"voi_name": "rib_right_7", "include": true},
    {"voi_name": "rib_right_8", "include": true},
    {"voi_name": "rib_right_9", "include": true},
    {"voi_name": "rib_right_10", "include": true},
    {"voi_name": "rib_right_11", "include": true},
    {"voi_name": "rib_right_12", "include": true},
    {"voi_name": "scapula_left", "include": true},
    {"voi_name": "scapula_right", "include": true},
    {"voi_name": "sternum", "include": true},
    {"voi_name": "kidney_left", "include": true, "new_name": "L Kidney"},
    {"voi_name": "kidney_right", "include": true, "new_name": "R Kidney"},
    {"voi_name": "liver", "include": true, "new_name": "Liver"},
    {"voi_name": "spleen", "include": true, "new_name": "Spleen"}
  ],
  "combine": [
    {
      "combined_voi_name": "Ribs + Sternum + Scapulae",
      "sources": [
        "rib_left_1", "rib_left_2", "rib_left_3", "rib_left_4",
        "rib_left_5", "rib_left_6", "rib_left_7", "rib_left_8",
        "rib_left_9", "rib_left_10", "rib_left_11", "rib_left_12",
        "rib_right_1", "rib_right_2", "rib_right_3", "rib_right_4",
        "rib_right_5", "rib_right_6", "rib_right_7", "rib_right_8",
        "rib_right_9", "rib_right_10", "rib_right_11", "rib_right_12",
        "scapula_left", "scapula_right", "sternum"
      ]
    }
  ]
}
```

This configuration creates an RT-STRUCT with:
- **Ribs + Sternum + Scapulae** (combined into a single ROI)
- **L Kidney**
- **R Kidney**
- **Liver**
- **Spleen**

### Now convert the masks to RT-STRUCT

In [7]:
# Convert with the simple config (kidneys + liver only)
rtstruct_result = convert_masks_to_rtstruct(
    segmentation_base_dir=str(totalsegmentator_output_dir),
    ct_series_paths=seg_result["ct_paths"],
    rtstruct_output_dir=str(rtstruct_output_dir ),
    config_path='./total_seg_config.json',  # Config file specifying which structures to include
)

No config found, adding all masks
RT-STRUCT: patient=ANON54121 timepoint=scan1 -> my_dosimetry_project/rtstructs/ANON54121/rtstruct_scan1.dcm
[INFO]: ROI mask is empty
Added ROI: prostate
Added ROI: gluteus_maximus_left
[INFO]: ROI mask is empty
Added ROI: vertebrae_C5
Added ROI: iliopsoas_left
[INFO]: ROI mask is empty
Added ROI: vertebrae_T1
Added ROI: rib_right_7
Added ROI: vertebrae_T12
Added ROI: atrial_appendage_left
Added ROI: gluteus_medius_right
Added ROI: lung_upper_lobe_right
Added ROI: autochthon_right
Added ROI: iliac_vena_right
[INFO]: ROI mask is empty
Added ROI: rib_left_1
Added ROI: scapula_right
Added ROI: rib_left_10
[INFO]: ROI mask is empty
Added ROI: humerus_right
[INFO]: ROI mask is empty
Added ROI: skull
Added ROI: lung_lower_lobe_left
Added ROI: rib_left_12
Added ROI: hip_right
Added ROI: portal_vein_and_splenic_vein
Added ROI: aorta
[INFO]: ROI mask is empty
Added ROI: rib_left_3
Added ROI: vertebrae_L4
[INFO]: ROI mask is empty
Added ROI: brachiocephalic_trun

# Step 5: Inspect the RT-STRUCT Files

PyTheranostics includes utilities to inspect RT-STRUCT contents:

In [9]:
from pytheranostics.segmentation import get_rtstruct_roi_names, print_rtstruct_info

# Get the first RT-STRUCT file path
patient_id = list(rtstruct_result.keys())[0]
timepoint = list(rtstruct_result[patient_id].keys())[0]
rtstruct_path = rtstruct_result[patient_id][timepoint]

print(f"Inspecting: {rtstruct_path}\n")
print_rtstruct_info(str(rtstruct_path))

# Get just the ROI names
roi_names = get_rtstruct_roi_names(str(rtstruct_path))
print(f"\nROI names: {roi_names}")

Inspecting: my_dosimetry_project/rtstructs/ANON54121/rtstruct_scan1.dcm


RT-STRUCT: rtstruct_scan1.dcm
Patient Name: DOE^JOHN
Patient ID: ANON54121
Study Date: 20181115
Structure Set Label: RTstruct

Number of ROIs: 117

ROI List:
  1. [1] prostate
  2. [2] gluteus_maximus_left
  3. [3] vertebrae_C5
  4. [4] iliopsoas_left
  5. [5] vertebrae_T1
  6. [6] rib_right_7
  7. [7] vertebrae_T12
  8. [8] atrial_appendage_left
  9. [9] gluteus_medius_right
  10. [10] lung_upper_lobe_right
  11. [11] autochthon_right
  12. [12] iliac_vena_right
  13. [13] rib_left_1
  14. [14] scapula_right
  15. [15] rib_left_10
  16. [16] humerus_right
  17. [17] skull
  18. [18] lung_lower_lobe_left
  19. [19] rib_left_12
  20. [20] hip_right
  21. [21] portal_vein_and_splenic_vein
  22. [22] aorta
  23. [23] rib_left_3
  24. [24] vertebrae_L4
  25. [25] brachiocephalic_trunk
  26. [26] femur_right
  27. [27] femur_left
  28. [28] vertebrae_C7
  29. [29] vertebrae_T10
  30. [30] rib_right_5
  31. [31] subcla

## Summary

This tutorial demonstrated:

1. **TotalSegmentator Integration**: PyTheranostics provides a streamlined interface to TotalSegmentator for automatic CT segmentation of 104 anatomical structures
   
2. **Two-Step Workflow** (Recommended):
   - `totalseg_segment()`: Run segmentation once (slow)
   - `convert_masks_to_rtstruct()`: Convert to RT-STRUCT multiple times with different configs (fast)
   
3. **Configuration Files**: Control which organs to include, rename them, and combine related structures
   
4. **RT-STRUCT Output**: Standard DICOM format compatible with treatment planning systems and DICOM viewers

### Key Advantages

- **Efficiency**: Segment once, create multiple RT-STRUCTs with different organ selections
- **Flexibility**: Easy to customize organ names and groupings for different clinical workflows
- **Automation**: Automatic CT discovery, patient ID extraction, and timepoint handling
- **Standardization**: DICOM RT-STRUCT output works with existing clinical tools

### Next Steps

- **Dosimetry**: Use these RT-STRUCTs for organ dose calculations
- **Visualization**: Load RT-STRUCTs in MIM, 3D Slicer, or treatment planning systems
- **Analysis**: Extract organ volumes, statistics, or use for dose-volume histogram analysis

### Citations

**TotalSegmentator**:
> Wasserthal J, Breit HC, Meyer MT, et al. TotalSegmentator: Robust Segmentation of 104 Anatomic Structures in CT Images. *Radiology: Artificial Intelligence*. 2023;5(5):e230024. https://doi.org/10.1148/ryai.230024

**Example Dataset**:
> Dewaraja Yuni and Van, Benjamin. University of Michigan Deep Blue Repository. SNMMI Dosimetry Challenge Dataset. https://doi.org/10.7302/864r-tb45